In [ ]:
from useful.helpers import *
%matplotlib tk
seed = 42
jax.config.update("jax_enable_x64", True)
key = jax.random.PRNGKey(seed)

wigner_function_from_inference, t, f = unpickle_me_this("/Users/iason/PycharmProjects/STRAIN/phase_II/wigner_result_pipe_2.pickle")

def tukey_window_this(mat, alpha=0.2):
    mat = np.asarray(mat)

    if mat.ndim == 1:
        # 1D case
        w = tukey(mat.size, alpha=alpha)
        return mat * w
    elif mat.ndim == 2:
        # 2D case
        ny, nx = mat.shape
        wx = tukey(nx, alpha=alpha)
        wy = tukey(ny, alpha=alpha)
        window2d = wy[:, None] * wx[None, :]
        return mat * window2d
    else:
        raise ValueError("Input must be 1D or 2D array")

wigner_function_from_inference = jnp.fft.fftshift(wigner_function_from_inference)
f = jnp.fft.fftshift(f)

wigner = tukey_window_this(smooth_matrix(wigner_function_from_inference, 5))

## 2D power spectrum



In [ ]:
import numpy as np

def power_spectrum_2d(image, detrend=True):
    img = image.astype(float)

    if detrend:
        img -= img.mean()

    ft = np.fft.fft2(img)
    ps2d = np.abs(ft)**2

    return ps2d


In [ ]:
ps_samples = []
white_noise_matrices = []
for i in range(1):
    white_noise = np.random.standard_normal(len(f))
    white_noise_stress, _, _ = Stress_re(white_noise, time=t, supress_print=True)
    white_noise_stress = white_noise_stress.real
    print(f"Calculated {i}th white noise stress")

    # smooth_white_stress = tukey_window_this(smooth_matrix(white_noise_stress, smoothing_lvl=5, mode="gaussian"))
    # white_stress = tukey_window_this(white_noise_stress)
    white_stress = tukey_window_this(wigner_function_from_inference.real)
    # ps_sample = power_spectrum_2d(smooth_white_stress)
    ps_sample = power_spectrum_2d(white_stress)

    # white_noise_matrices.append(smooth_white_stress)
    white_noise_matrices.append(white_stress)
    ps_samples.append(ps_sample)

In [ ]:
noise_ps = np.mean(ps_samples, axis=0)
# cfm_noise_ps = power_spectrum_2d(tukey_window_this(cfm_samples[0]))

In [ ]:
def sample_from_ps2d(psd):
    ny, nx = psd.shape
    vol = np.sqrt(nx)
    xi = np.random.normal(size=(ny, nx)) / vol # standard normal
    field = np.fft.ifft2(np.sqrt(psd) * xi, norm="forward").real
    return field

def average_xi_dot(N=10, shape=(256, 256), data_type=np.float64):
    acc = np.zeros((shape[0], shape[0]), dtype=data_type)
    vol = shape[0]
    for _ in range(N):
        xi = np.random.normal(size=shape, scale=1, loc=0) / np.sqrt(vol)
        # xi_ft = np.fft.fft2(xi)
        acc += xi @ xi.T
        # acc += xi_ft @ xi_ft.T
    return (acc / N).real

# average_xi_dot(N=500, shape=(100,100))


In [ ]:
sl = sample_from_ps2d(noise_ps)
# cfm_sl = sample_from_ps2d(cfm_noise_ps)

In [ ]:
# visualize_stress(np.log(noise_ps), rows=f, cols=t, smooth=False)
visualize_stress(sl, rows=np.array(list(range(len(f)))), cols=np.array(list(range(len(t)))), smooth=True)
# visualize_stress(white_noise_matrices[0], rows=np.array(list(range(len(f)))), cols=np.array(list(range(len(t)))), smooth=False)
# visualize_stress(cfm_sl, rows=np.array(list(range(len(f)))), cols=np.array(list(range(len(t)))), smooth=False)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# assume noise_ps is your 2D PSD
ny, nx = noise_ps.shape

# slices
kx_slice = noise_ps[ny//2, :]   # middle row → kx slice
ky_slice = noise_ps[:, nx//2+10]   # middle column → ky slice

plt.figure()
plt.plot(kx_slice, label='kx slice (middle row)')
plt.plot(ky_slice, label='ky slice (middle column)')
plt.xlabel('Index (Fourier mode)')
plt.ylabel('Power')
plt.yscale('log')   # log scale often helps
plt.legend()
plt.title('1D slices of 2D PSD')
plt.show()


In [ ]:
cfm_maker = jft.CorrelatedFieldMaker(prefix="s_")
cfm_maker.set_amplitude_total_offset(0, (1e-16,1e-16))
n_t = len(t)
n_f = len(f)
# dist_t = 1
dist_t = t[1]-t[0]
# dist_f = 1
dist_f = f[1]-f[0]


In [ ]:
cfm_maker.add_fluctuations(shape=(n_f,n_t), distances=(1, 1), fluctuations=(10,5), flexibility=(1,1),
                           # asperity=(3,2),
                                   loglogavgslope=(-4,1), harmonic_type="fourier",
                                   non_parametric_kind="power")

In [ ]:
s = cfm_maker.finalize()

In [ ]:
cfm_samples = []
for i in range(1):
    key, key_s = jax.random.split(key)
    xi = jft.random_like(key_s, s.domain)
    sl = s(xi)
    cfm_samples.append(sl)

In [ ]:
visualize_stress(cfm_samples[0], rows=np.array(list(range(len(f)))), cols=np.array(list(range(len(t)))), smooth=False)

In [ ]:
sig=.1
N_inv = lambda xi: xi/sig**2
lh = jft.Gaussian(data=jnp.array(white_noise_matrices[0]), noise_cov_inv=N_inv).amend(s)

In [ ]:
key, key_sampler, key_i = jax.random.split(key, 3)

In [ ]:
linear_loose=(0.02, 100)
non_linear_loose=(0.5, 20)
kl_loose=(0.1, 35)

linear_energy, linear_iter = linear_loose
nonlinear_energy, nonlinear_iter = non_linear_loose
kl_energy, kl_iter = kl_loose

draw_linear_kwargs = dict(
            cg_name="linear_sampler",
            cg_kwargs=dict(absdelta=linear_energy, maxiter=linear_iter),
        )

# Arguments for the minimizer in the nonlinear updating of the samples
nonlinearly_update_kwargs = dict(
    minimize_kwargs=dict(
                name="non_linear_sampler",
                xtol=nonlinear_energy,
                cg_kwargs=dict(name=None),
                maxiter=nonlinear_iter,
            )
        )

# Arguments for the minimizer of the KL-divergence cost potential
kl_kwargs = dict(
            minimize_kwargs=dict(
                name="kl_minimizer", xtol=kl_energy, cg_kwargs=dict(name=None), maxiter=kl_iter
            )
        )

post_samples, _ = jft.optimize_kl(
            likelihood=lh,
            position_or_samples=jft.Vector(lh.init(key_i)),
            key=key_sampler,
            n_total_iterations=1,
            n_samples=15,
            draw_linear_kwargs=draw_linear_kwargs,
            nonlinearly_update_kwargs=nonlinearly_update_kwargs,
            kl_kwargs=kl_kwargs,
            sample_mode="linear_resample",
            resume=False,
            odir=None,
            callback=None

        )

In [ ]:
signal_vals = [s(p_sl) for p_sl in post_samples]
mean_signal = jft.mean(jnp.array(signal_vals))

In [ ]:
list(post_samples)

## Direct inversion

Let $n$ be the 2d windowed noise:

$$ n_{xy} = (F^{-1} \sqrt{p_n} \Xi)_{xy},$$

where $\Xi$ is 3d iid. Then,

$$ \sqrt{p_n}^{xy} = (\tilde{n} \Xi^{-1})^{xy},$$

since $\Xi$ is invertible almost surely.

In [ ]:
def get_asd_with_force(image, itr=1):
    N = image.shape[0]
    image_ft = np.fft.fft2(image)

    asd2d_samples = []

    for _ in range(itr):
        Xi = np.random.standard_normal(size=(N, N))

        try:
            Xi_inv = np.linalg.inv(Xi)
        except np.linalg.LinAlgError:
            raise ValueError("matrix was not invertible, try again")

        asd2d_sample = image_ft @ Xi_inv
        asd2d_samples.append(asd2d_sample)

    asd2d = np.mean(np.array(asd2d_samples), axis=0)
    return asd2d, asd2d_samples


In [ ]:
whi